# VocalLift — fine-tune a singing engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AaronGrace978/Voice-Enhancer/blob/v0.1.0/notebooks/VocalLift_Colab.ipynb)

VocalLift does not write a song from a prompt. It takes a voice that already
knows the tune and helps it land the notes: cracks get rebuilt, the melody
moves into the key (or onto a guide track), and the vibrato stays.

This notebook is the training half of that idea.

1. The signal engine runs with no training at all. It is a vocoder, a pitch repair, and a retune.
2. A small differentiable synthesizer then learns to *sing the phrase back* after you have damaged it: shifted, cracked, noised, dropped out.
3. Optionally, that model fine-tunes on recordings of one voice, so the resynthesis picks up their tone.

Run it on a GPU if you have one. A CPU is enough to see the loss fall.


## Setup

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "vocallift").exists() and (ROOT.parent / "vocallift").exists():
    os.chdir(ROOT.parent)
    ROOT = Path.cwd()
if not (ROOT / "vocallift").exists():
    dest = "/content/Voice-Enhancer" if Path("/content").exists() else str(Path.cwd() / "Voice-Enhancer")
    if not (Path(dest) / "vocallift").exists():
        # $dest is expanded by IPython. If this branch is gone after merge, clone main.
        !git clone --branch v0.1.0 --single-branch https://github.com/AaronGrace978/Voice-Enhancer.git "$dest" || (rm -rf "$dest" && git clone https://github.com/AaronGrace978/Voice-Enhancer.git "$dest")
    os.chdir(dest)
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

!pip install -q "setuptools>=69,<81" "numpy>=1.26" "scipy>=1.11" "soundfile>=0.12" "soxr>=0.3" "av>=12" "pyworld>=0.3.4" "matplotlib>=3.8"

import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("repo:", ROOT)


## Hear the signal engine first

No checkpoint, no GPU. A synthesized phrase is sung on purpose about 40 cents
out of tune, with an octave crack and a short dropout. The engine's job is to
put the same melody back on the pitches, not to replace the singer.


In [ ]:
import numpy as np
from IPython.display import Audio, display
import matplotlib.pyplot as plt

from vocallift.demo import MELODY, flat_take, guide_take, offkey_take
from vocallift.music import note_name
from vocallift.pipeline import EnhanceSettings, enhance

take = offkey_take()
result = enhance(take, settings=EnhanceSettings(key="C", scale="major", strength=1, tightness=0.55, retune_ms=35, reverb=0.12))
print(result.summary())
print("melody:", " ".join(note_name(m) for m in MELODY))
print("targets:", " ".join(note_name(n.target) for n in result.correction.notes))

print("sung")
display(Audio(take, rate=44100))
print("enhanced")
display(Audio(result.audio, rate=result.sr))

c = result.correction
t = np.arange(len(c.midi_in)) * result.hop / result.sr
plt.figure(figsize=(10, 3.2))
plt.plot(t, np.where(c.voiced, c.midi_in, np.nan), color="#e07a5f", label="sung", linewidth=1.4)
plt.plot(t, np.where(c.voiced, c.midi_out, np.nan), color="#6a8f2a", label="after", linewidth=1.8)
plt.ylabel("MIDI pitch")
plt.xlabel("seconds")
plt.title(f"{result.before_cents:.0f}¢ off  →  {result.after_cents:.0f}¢")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


## When the nearest note is the wrong note

Snap-to-scale can only move you to the closest tone in the key. A minor third
flat is closer to a different note, so the nearest-note rule lands in the
wrong place. Hand the engine a guide — the tune, sung or played — and each
note of the performance inherits the guide's pitch while keeping the singer's timing.


In [ ]:
flat = flat_take()
guide = guide_take()
pulled = enhance(
    flat,
    settings=EnhanceSettings(key="C", scale="major", strength=1, tightness=0.5, retune_ms=30, reverb=0.1),
    guide=guide,
)
print(pulled.summary())
print("targets:", " ".join(note_name(n.target) for n in pulled.correction.notes))
print("sung, a long way flat")
display(Audio(flat, rate=44100))
print("the melody it was meant to be")
display(Audio(guide, rate=44100))
print("pulled onto that melody")
display(Audio(pulled.audio, rate=pulled.sr))


## What the network is learning

The signal engine already decided the pitch. The network's job is narrower,
and it is the part you fine-tune:

* Build a clean sung phrase with a differentiable harmonic-plus-noise synthesizer.
* Damage a copy: shift it by up to a couple of semitones, throw in a brief octave crack, add noise, punch out a short hole.
* Show the network the damaged audio and the *clean* pitch.
* Ask it to sing the clean phrase.

It has to read the vowel off the damaged recording and follow the pitch it was given. That is the same job it will have in the booth, where the pitch it is given is the corrected melody.

This is self-supervised. You do not need a dataset of "bad take / good take" pairs to start. Those pairs — a rough vocal against a cleaner double of the same person — are what you add in the fine-tune cell, and they teach the model that voice.


In [ ]:
from vocallift.neural.model import pick_device
from vocallift.neural.train import finetune_files, save_checkpoint, train_steps

device = pick_device()
# A free CPU run is short so the notebook finishes. On a GPU, a few hundred
# steps is where the resynthesis starts to hold a vowel.
STEPS = 200 if device.type == "cuda" else 60
BATCH = 16 if device.type == "cuda" else 4

model, losses = train_steps(steps=STEPS, batch=BATCH, seconds=1.2, device=device, log_every=10)
save_checkpoint(model, "checkpoints/vocallift.pt")
print(f"loss {losses[0]:.3f} → {losses[-1]:.3f}")

plt.figure(figsize=(8, 3))
plt.plot(losses, color="#c4622d")
plt.xlabel("step")
plt.ylabel("spectral loss")
plt.title("Pretrain on damaged synthetic singing")
plt.tight_layout()
plt.show()


## Fine-tune on a voice

Set `FINETUNE = True` and run the cell. In Colab it asks for a few dry vocal
takes — scales, a verse, anything where the voice is the only thing in the
recording. A minute or two is enough to hear the model move. More is better.

Each step reconstructs the recording from its own pitch, then does it again
from a noisy copy. The checkpoint is overwritten with the adapted model.


In [ ]:
FINETUNE = False

if not FINETUNE:
    print("Set FINETUNE = True, then run this cell and upload dry vocal takes.")
else:
    try:
        from google.colab import files
        print("Upload wav or mp3. Dry voice, no backing track.")
        uploaded = files.upload()
        paths = list(uploaded)
    except ImportError:
        paths = []  # or a list of local file paths
        print("Not in Colab. Set `paths` to your recordings and re-run.")
    if paths:
        finetune_losses = finetune_files(model, paths, steps=80, device=device)
        save_checkpoint(model, "checkpoints/vocallift.pt")
        print(f"finetune loss {finetune_losses[0]:.3f} → {finetune_losses[-1]:.3f}")


## Resynthesize at the corrected pitch

With a checkpoint in place, the booth runs the signal engine first (repair,
key, vibrato) and then asks the network to sing that corrected pitch in the
tone it heard. The consonants of the original take are kept.


In [ ]:
neural = enhance(
    take,
    settings=EnhanceSettings(
        key="C",
        scale="major",
        strength=1,
        retune_ms=35,
        reverb=0.08,
        neural=True,
        checkpoint="checkpoints/vocallift.pt",
    ),
)
print(neural.summary())
print("signal engine")
display(Audio(result.audio, rate=result.sr))
print("signal engine, then neural resynthesis")
display(Audio(neural.audio, rate=neural.sr))


## Take it back to the booth

Download `checkpoints/vocallift.pt` and put it in the same place in the repo
on your machine (or set `VOCALLIFT_CHECKPOINT` to the file). Then:

```bash
python -m vocallift serve
```

Open the booth, enable **Neural resynth**, and run a take. The signal path
keeps working if the checkpoint is missing; the network is an extra voice,
not a dependency.


In [ ]:
try:
    from google.colab import files
    files.download("checkpoints/vocallift.pt")
except ImportError:
    print("Checkpoint saved at checkpoints/vocallift.pt")
